In [1]:
import torch
import numpy as np
from PIL import Image

def image_tensor(image_path, size=None):
    img = Image.open(image_path)
    
    if size != None:
        img = img.resize(size)

    img = (np.asarray(img)/255.0)
    img = torch.from_numpy(img).float()
    img = img.permute(2,0,1)
    img = img.cuda().unsqueeze(0)
    return img

In [2]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities


/home/soom/miniconda3/envs/clode/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
probs

tensor([[9.9925e-01, 7.5487e-04]], grad_fn=<SoftmaxBackward0>)

In [ ]:
from pathlib import Path
data_path = Path('/home/soom/data/LOL/eval15')

from transformers import CLIPVisionModel, CLIPTextModel, CLIPProcessor
clip_text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")
clip_text_encoder.eval()
clip_vision_encoder = CLIPVisionModel.from_pretrained("openai/clip-vit-large-patch14")
clip_vision_encoder.eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

In [5]:
from transformers import AutoTokenizer, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-large-patch14")

inputs = tokenizer(["a photo of a cat", "a photo of a dog"], padding=True, return_tensors="pt")
text_features = model.get_text_features(**inputs)

In [19]:
text_features.shape

torch.Size([2, 768])

In [ ]:
outputs = clip_text_encoder(**inputs)
pooled_output = outputs.pooler_output
text_features2 = model.text_projection(pooled_output)

print(outputs.last_hidden_state.shape)
print(pooled_output.shape)

In [ ]:
torch.allclose(text_features, text_features2, atol=1e-6)

In [ ]:
image_features = model.get_image_features(**inputs)

In [8]:
img_t = image_tensor('cat.jpg', size=(224, 224))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
from torchmetrics.multimodal import CLIPImageQualityAssessment

clip_metric = CLIPImageQualityAssessment(
    model_name_or_path="openai/clip-vit-large-patch14",
    prompts=('brightness', 'noisiness', 'quality', 'colorfullness', 'contrast', 'complexity', 'warm')
)
    
clip_vision_encoder = clip_metric.model.vision_model
clip_vision_encoder.eval()

clip_visual_projection = clip_metric.model.visual_projection
clip_visual_projection.eval()

Linear(in_features=1024, out_features=768, bias=False)

In [10]:
img_feat = clip_vision_encoder(img_t.detach().cpu())
img_feat_t = clip_visual_projection(img_feat[1]) 

In [11]:
img_feat_t.shape

torch.Size([1, 768])

In [13]:
img_feat_t2 = model.get_image_features(img_t.detach().cpu())
img_feat_t2.shape

torch.Size([1, 768])

In [14]:
torch.allclose(img_feat_t, img_feat_t2, atol=1e-6)

True

In [16]:
import numpy as np

text_emb_bw = np.load('/home/lbw/CLODE/traindata_csv/text_embedding_pca.npy')
text_emb_sm = np.load('/home/soom/CLODE2/interval_predictor/data/LOL/train_clip_features_v2.npy')

In [17]:
image_emb_bw = np.load('/home/lbw/CLODE/traindata_csv/vision_cls_pca.npy')
image_emb_bw.shape


(485, 512)

In [10]:
from PIL import Image
def image_tensor(image_path, size=None):
    img = Image.open(image_path)
    
    if size != None:
        img = img.resize(size)

    img = (np.asarray(img)/255.0)
    img = torch.from_numpy(img).float()
    img = img.permute(2,0,1)
    img = img.cuda().unsqueeze(0)
    return img

In [19]:
from transformers import AutoTokenizer, CLIPModel
import os
from pathlib import Path
import torch
from PIL import Image
import torchvision.transforms as transforms

data_path = Path('/home/soom/data/LOL/our485')
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to('cuda')
image_labels = sorted(os.listdir(data_path / 'low'))

transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                                std=[0.26862954, 0.26130258, 0.27577711])
        ])

with torch.no_grad():
    model.eval()
    image_features = []
    for label in image_labels:            
        image = Image.open(data_path / 'low' / label).convert('RGB')
        lq_224 = transform(image).unsqueeze(0).to('cuda')
        image_feat = model.get_image_features(lq_224)
        image_features.append(image_feat.squeeze(0))    
    image_features = torch.stack(image_features)
    image_emb_sm = image_features.cpu().numpy()
    
image_emb_sm.shape

(485, 512)

In [20]:

np.allclose(image_emb_bw, image_emb_sm, atol=1e-6)

True

In [2]:
text_emb_bw.shape

(3, 512)

In [3]:
text_emb_sm.shape

(485, 4, 512)

In [4]:
text_emb_sm[0, :-1, :].shape

(3, 512)

In [5]:
np.allclose(text_emb_bw, text_emb_sm[0, :-1, :], atol=1e-6)

True